# Extraccion de caracteristicas
![Logo](itq.jpg)
#### Nombre: Santiago Carrasco  
#### Fecha: 25/6/2026
https://github.com/Santi2w/pythonintroduccion2/blob/main/Programa15.Clasificacion.LOGR.ipynb

In [ ]:
# Instalamos las librerías necesarias para leer señales ECG del dataset MIT-BIH.
!pip install wfdb scipy
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy import stats
from scipy.signal import butter, filtfilt
from scipy.fft import rfft, rfftfreq

# Parte I: Extracción
En esta parte se obtiene el dataset , se revisa su estructura y se carga una primera señal ECG

## Descompresión del dataset
Esta celda indica la ruta del archivo zip, define la carpeta donde se extraerá y descomprime todos los archivos necesarios para leer los registros ECG.

In [ ]:
# Guardamos en una variable el nombre del archivo comprimido del dataset.
zip_path = "mit-bih-arrhythmia-database-1.0.0.zip"

extract_path = "mit_bih_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraído correctamente")

## Búsqueda de registros ECG
Recorremos todas las carpetas del dataset y busca archivos con extensión hea. Estos archivos son los registros WFDB y permiten identificar cada señal disponible.

In [ ]:
# Creamos una lista vacía para guardar las rutas de los registros ECG encontrados.
registros = []

for root, dirs, files in os.walk("mit_bih_dataset"):
    for file in files:
        if file.endswith(".hea"):
            ruta = os.path.join(root, file).replace(".hea", "")
            
            registros.append(ruta)

print("Número de registros encontrados:", len(registros))

print("Primeros registros:")
print(registros[:5])

## Lectura del primer registro
Tomamos el primer registro encontrado, lo lee con wfdb.rdrecord() y extrae el primer canal de la señal ECG. También obtiene la frecuencia de muestreo, el número de muestras y los nombres de los canales.

In [ ]:
# Seleccionamos el primer registro encontrado para hacer pruebas iniciales.
registro = registros[0]

record = wfdb.rdrecord(registro)

senal = record.p_signal[:, 0]

fs = record.fs

print("Registro:", registro)
print("Frecuencia de muestreo:", fs)
print("Número de muestras:", len(senal))
print("Canales:", record.sig_name)

## Visualización de la señal original
En esta celda grafica las primeras 2000 muestras de la señal ECG original para observar su forma antes de aplicar cualquier transformación.

In [ ]:
# Creamos una figura para graficar la señal con tamaño ancho.
plt.figure(figsize=(12, 4))

plt.plot(senal[:2000])

plt.title("Señal ECG original")
plt.xlabel("Muestras")
plt.ylabel("Amplitud")

plt.grid()

plt.show()

# Parte II: Exploración
Se revisa la calidad básica de la señal: cantidad de registros, número de muestras, valores faltantes, tipo de dato, valores anómalos y ruido visual.

## Revisión de calidad de datos
Se muestra información básica de la señal cargada, como cantidad de registros, tamaño de la señal, valores faltantes y tipo de variable.

In [ ]:
# Mostramos cuántos registros ECG fueron encontrados en el dataset.
print("Número de registros:", len(registros))

print("Número de muestras por señal:", len(senal))

print("Valores faltantes:")
print(np.isnan(senal).sum())

print("Tipo de variable:")
print(type(senal[0]))

## Detección de valores anómalos
calculamos la media y la desviación estándar. Luego cuenta cuántos valores están alejados más de tres desviaciones estándar, lo cual permite detectar posibles valores extremos.

In [ ]:
# Calculamos la media de la señal para conocer su valor promedio.
media = np.mean(senal)

desviacion = np.std(senal)

anomalias = np.sum(np.abs(senal - media) > 3 * desviacion)

print("Media:", media)
print("Desviación estándar:", desviacion)
print("Valores anómalos:", anomalias)

## Observación del ruido
Esta celda vuelve a graficar un fragmento de la señal para revisar visualmente si existe ruido. Al final imprime una observación general sobre el nivel de ruido.

In [ ]:
# Creamos una figura para observar un fragmento de la señal.
plt.figure(figsize=(12, 4))

plt.plot(senal[:2000])

plt.title("Observación de ruido en la señal ECG")
plt.xlabel("Muestras")
plt.ylabel("Amplitud")

plt.grid()

plt.show()

print("Nivel de ruido observado: leve o moderado según la forma de la señal")

# Parte III: Transformación
Preparamos los datos para el análisis. Se elimina el offset, se normaliza, se filtra el ruido, se escala la señal y se comparan los resultados.

## Eliminación del offset
Se resta la media a toda la señal para centrarla alrededor de cero. Esto elimina el desplazamiento vertical que puede tener la señal original.

In [ ]:
# Calculamos una nueva señal restando la media de la señal original.
senal_sin_offset = senal - np.mean(senal)

print("Media original:", np.mean(senal))

print("Media después de quitar offset:", np.mean(senal_sin_offset))

## Normalización de la señal
Esta celda convierte los valores de la señal a una escala entre 0 y 1. Esto ayuda a comparar señales que pueden tener amplitudes diferentes.

In [ ]:
# Normalizamos la señal para que sus valores queden entre 0 y 1.
senal_normalizada = (
    senal_sin_offset - np.min(senal_sin_offset)
) / (
    np.max(senal_sin_offset) - np.min(senal_sin_offset)
)

print("Mínimo:", np.min(senal_normalizada))

print("Máximo:", np.max(senal_normalizada))

## Filtrado pasa banda
Define una función para aplicar un filtro pasa banda entre 0.5 Hz y 40 Hz. El objetivo es conservar la información importante del ECG y reducir ruido de baja y alta frecuencia.

In [ ]:
# Importamos nuevamente las funciones necesarias para el filtrado.
from scipy.signal import butter, filtfilt

def filtro_pasabanda(signal, fs, lowcut=0.5, highcut=40, orden=4):
    nyquist = 0.5 * fs

    low = lowcut / nyquist
    
    high = highcut / nyquist

    b, a = butter(orden, [low, high], btype="band")

    return filtfilt(b, a, signal)

senal_filtrada = filtro_pasabanda(senal_sin_offset, fs)

print("Señal filtrada correctamente")

## Escalamiento con Z-Score
Transformamos la señal filtrada para que tenga media cercana a 0 y desviación estándar cercana a 1. Esto ayuda a estandarizar los datos.

In [ ]:
# Escalamos la señal usando Z-Score.
senal_escalada = (
    senal_filtrada - np.mean(senal_filtrada)
) / np.std(senal_filtrada)

print("Media escalada:", np.mean(senal_escalada))

print("Desviación escalada:", np.std(senal_escalada))

## Conversión a DataFrame
Esta celda organiza en una tabla las diferentes versiones de la señal: original, sin offset, normalizada, filtrada y escalada.

In [ ]:
# Creamos un DataFrame para guardar todas las versiones de la señal.
df_senal_transformada = pd.DataFrame({
    "señal_original": senal,
    
    "señal_sin_offset": senal_sin_offset,
    
    "señal_normalizada": senal_normalizada,
    
    "señal_filtrada": senal_filtrada,
    
    "señal_escalada": senal_escalada
})

df_senal_transformada.head()

## Comparación de señales transformadas
Graficamos la señal original, filtrada y escalada para comparar visualmente cómo cambió la señal después del procesamiento.

In [ ]:
# Creamos una figura para comparar varias señales en la misma gráfica.
plt.figure(figsize=(12,4))

plt.plot(senal[:2000], label="Original")

plt.plot(senal_filtrada[:2000], label="Filtrada")

plt.plot(senal_escalada[:2000], label="Escalada")

plt.title("Comparación de señales ECG transformadas")
plt.xlabel("Muestras")
plt.ylabel("Amplitud")

plt.legend()

plt.grid()

plt.show()

# Parte IV: Extracción de características
En esta parte se calculan valores numéricos que resumen el comportamiento de la señal ECG. Estas características pueden usarse después para análisis o modelos de machine learning.

## Librerías para características
Esta celda importa herramientas estadísticas y de transformada rápida de Fourier para calcular características en el dominio del tiempo y de la frecuencia.

In [ ]:
# Importamos funciones estadísticas para calcular características de la señal.
from scipy import stats

from scipy.fft import rfft, rfftfreq

## Función para extraer características
Se crea una función que recibe una señal y su frecuencia de muestreo. La función calcula características estadísticas, de forma, energía, temporales y frecuenciales, y las devuelve en un diccionario.

In [ ]:
# Definimos una función para extraer características numéricas de una señal ECG.
def extraer_caracteristicas(signal, fs):

    caracteristicas = {}


    caracteristicas["media"] = np.mean(signal)

    caracteristicas["mediana"] = np.median(signal)

    caracteristicas["moda"] = stats.mode(signal, keepdims=True)[0][0]

    caracteristicas["maximo"] = np.max(signal)

    caracteristicas["minimo"] = np.min(signal)

    caracteristicas["varianza"] = np.var(signal)

    caracteristicas["desviacion_estandar"] = np.std(signal)

    caracteristicas["rango"] = np.max(signal) - np.min(signal)

    caracteristicas["percentil_25"] = np.percentile(signal, 25)

    caracteristicas["percentil_75"] = np.percentile(signal, 75)


    caracteristicas["curtosis"] = stats.kurtosis(signal)

    caracteristicas["asimetria"] = stats.skew(signal)


    caracteristicas["rms"] = np.sqrt(np.mean(signal ** 2))

    caracteristicas["energia"] = np.sum(signal ** 2)

    caracteristicas["potencia"] = np.sum(signal ** 2) / len(signal)


    caracteristicas["cruces_por_cero"] = np.sum(
        np.diff(np.sign(signal)) != 0
    )

    caracteristicas["pico_maximo"] = np.max(signal)

    caracteristicas["pico_minimo"] = np.min(signal)


    fft_valores = np.abs(rfft(signal))

    fft_frecuencias = rfftfreq(len(signal), 1 / fs)

    caracteristicas["frecuencia_dominante"] = (
        fft_frecuencias[np.argmax(fft_valores)]
    )

    caracteristicas["magnitud_maxima"] = np.max(fft_valores)

    caracteristicas["energia_espectral"] = np.sum(
        fft_valores ** 2
    )

    return caracteristicas

## Aplicación de la función a una señal
Usamos la función extraer_caracteristicas() sobre la señal filtrada y guarda el resultado en la variable `caracteristicas`.

In [ ]:
# Aplicamos la función de extracción a la señal ECG filtrada.
caracteristicas = extraer_caracteristicas(
    senal_filtrada,
    
    fs
)

caracteristicas

## Tabla de características
En esta celda convierte el diccionario de características en un DataFrame para visualizar los resultados de manera ordenada.

In [ ]:
# Convertimos el diccionario de características en un DataFrame de pandas.
df_caracteristicas = pd.DataFrame(
    [caracteristicas]
)

pd.set_option('display.max_columns', None)

pd.options.display.float_format = '{:.4f}'.format

print("Características extraídas de la señal ECG:")

df_caracteristicas.T.rename(
    columns={0: "Valor"}
)

## Análisis frecuencial con FFT
Esta celda aplica la transformada rápida de Fourier para observar las frecuencias presentes en la señal ECG filtrada.

In [ ]:
# Calculamos la magnitud de la FFT de la señal filtrada.
fft_valores = np.abs(
    rfft(senal_filtrada)
)

fft_frecuencias = rfftfreq(
    len(senal_filtrada),
    1 / fs
)

plt.figure(figsize=(12,4))

plt.plot(
    fft_frecuencias,
    fft_valores
)

plt.title("FFT de la señal ECG")
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")

plt.grid()

plt.show()

# Parte V: Carga
En esta parte se procesan todos los registros encontrados, se extraen sus características y se guarda el resultado final en un archivo CSV.

## Procesamiento de todos los registros
Esta celda recorre cada registro del dataset. Para cada señal, elimina offset, filtra ruido, extrae características y guarda una fila con el nombre del registro.

In [ ]:
# Creamos una lista vacía donde se guardará una fila de características por cada registro.
datos = []

for registro in registros:

    try:

        record = wfdb.rdrecord(registro)

        signal = record.p_signal[:, 0]

        fs = record.fs

        signal = signal - np.mean(signal)

        signal = filtro_pasabanda(signal, fs)

        fila = extraer_caracteristicas(signal, fs)

        fila["registro"] = os.path.basename(registro)

        datos.append(fila)

    except Exception as e:
        print("Error en:", registro)
        print(e)

## Creación del dataset final
Convertimos la lista de características en un DataFrame final, donde cada fila representa un registro ECG procesado.

In [ ]:
# Convertimos la lista de diccionarios en un DataFrame final.
df_final = pd.DataFrame(datos)

print("Número de registros procesados:", len(df_final))

df_final.head()

## Verificación de dimensiones
Mostramos cuántas filas y columnas tiene el dataset final para confirmar la cantidad de registros y características generadas.

In [ ]:
# Mostramos la cantidad de filas del DataFrame final.
print("Filas:", df_final.shape[0])

print("Columnas:", df_final.shape[1])

## Guardado del archivo CSV
Esta celda exporta el DataFrame final a un archivo llamado `Caracteristicas_ecg.csv`, sin guardar el índice de pandas.

In [ ]:
# Guardamos el DataFrame final como archivo CSV.
df_final.to_csv(
    "Caracteristicas_ecg.csv",
    
    index=False
)

print("Archivo guardado correctamente")

## Vista previa del dataset generado
Se muestra las primeras filas del dataset final para revisar que las características se hayan generado correctamente.

In [ ]:
# Configuramos pandas para mostrar todas las columnas del DataFrame.
pd.set_option('display.max_columns', None)

df_final.head()

## Confirmación de archivo creado
Esta celda verifica si el archivo CSV existe en la carpeta de trabajo y muestra un mensaje de confirmación.

In [ ]:
# Importamos os para verificar si existe un archivo en la carpeta de trabajo.
import os

if os.path.exists("Caracteristicas_ecg.csv"):
    print("Caracteristicas_ecg.csv creado correctamente")
else:
    print("No se encontró el archivo")